In [3]:
# ============================================================
# 0. 실습 환경 확인
# ============================================================

# torch는 PyTorch의 핵심 라이브러리입니다.
# 텐서 연산, 자동 미분, 신경망 모델 작성에 사용합니다.
import torch

# torch.nn은 PyTorch에서 모델 계층, 손실 함수 등을 제공하는 모듈입니다.
import torch.nn as nn

# torch.nn.functional은 활성화 함수, 손실 함수 등을 함수 형태로 제공합니다.
import torch.nn.functional as F

# torch.utils.data는 Dataset, DataLoader를 제공하여 데이터를 미니배치로 나누어 학습할 수 있게 합니다.
from torch.utils.data import TensorDataset, DataLoader

# pandas는 CSV 파일을 읽고 표 형태 데이터프레임으로 다루기 위해 사용합니다.
import pandas as pd

# numpy는 수치 계산 및 배열 처리를 위해 사용합니다.
import numpy as np

# matplotlib은 학습 결과 그래프를 그리기 위해 사용합니다.
import matplotlib.pyplot as plt

# sklearn의 LabelEncoder는 문자 A~Z 같은 범주형 라벨을 0~25 숫자로 변환합니다.
from sklearn.preprocessing import LabelEncoder, StandardScaler

# sklearn의 confusion_matrix, classification_report는 분류 성능 평가에 사용합니다.
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from sklearn.model_selection import train_test_split

# os는 파일 경로 존재 여부를 확인하기 위해 사용합니다.
import os

# urllib.request는 Colab에서 데이터 파일이 없을 때 인터넷에서 데이터를 내려받기 위해 사용합니다.
import urllib.request

# random은 파이썬 기본 난수 생성을 제어하기 위해 사용합니다.
import random

# 실행 장치를 설정합니다.
# GPU가 있으면 cuda, 없으면 cpu를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 사용 중인 장치를 출력합니다.
print("사용 장치:", device)

사용 장치: cuda


In [2]:
# ============================================================
# 1. 재현 가능한 실험을 위한 시드 고정
# ============================================================

# 같은 코드를 여러 번 실행해도 가능한 한 비슷한 결과가 나오도록 난수 시드를 고정합니다.
SEED = 12345

# 파이썬 random 모듈의 난수 시드를 고정합니다.
random.seed(SEED)

# numpy 난수 시드를 고정합니다.
np.random.seed(SEED)

# PyTorch CPU 난수 시드를 고정합니다.
torch.manual_seed(SEED)

# GPU가 사용 가능한 경우 GPU 난수 시드도 고정합니다.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("난수 시드 고정 완료:", SEED)

난수 시드 고정 완료: 12345


In [3]:
# ============================================================
# 2. 데이터 파일 준비
# ============================================================

# Colab에서 사용할 데이터 폴더 경로를 지정합니다.
DATA_DIR = "/content/sample_data"
# 데이터 폴더가 없으면 생성합니다.
os.makedirs(DATA_DIR, exist_ok=True)

# 사용할 CSV 파일 경로를 지정합니다.
csv_path = os.path.join(DATA_DIR, "postcode.csv")

# CSV 파일이 없는 경우 UCI Letter Recognition 데이터를 내려받습니다.
if not os.path.exists(csv_path):
    # UCI Letter Recognition 데이터셋 URL입니다.
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"

    # 임시 원본 데이터 파일 경로입니다.
    raw_path = os.path.join(DATA_DIR, "letter-recognition.data")

    # 인터넷에서 데이터를 다운로드합니다.
    urllib.request.urlretrieve(url, raw_path)

    # UCI 데이터셋은 헤더가 없으므로 컬럼명을 직접 지정합니다.
    columns = [
        "letter", "x_box", "y_box", "width", "height", "onpix", "x_bar", "y_bar",
        "x2bar", "y2bar", "xybar", "x2ybr", "xy2br", "x_ege", "xegvy",
        "y_ege", "yegvx"
    ]

    # CSV 파일을 pandas DataFrame으로 읽습니다.
    df_raw = pd.read_csv(raw_path, header=None, names=columns)

    # CSV 파일 형태로 저장합니다.
    df_raw.to_csv(csv_path, index=False)

    print("데이터 다운로드 및 CSV 저장 완료:", csv_path)
else:
    print("기존 CSV 파일 사용:", csv_path)

# CSV 파일을 pandas DataFrame으로 읽습니다.
df = pd.read_csv(csv_path, encoding='cp949')

# 데이터 앞부분 5개 행을 출력하여 구조를 확인합니다.
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/postcode.csv'

In [ ]:
df["읍면동이름"] = df["읍면동이름"].fillna("없음")

# df['우편번호앞'] = df['우편번호'].astype(str).str[:3].astype(int)
df['우편번호앞'] = df['우편번호'].str[:3].astype(int)
df['우편번호뒤'] = df['우편번호'].astype(str).str[3:].astype(int)

df = df.drop(columns=['우편번호', '우편번호순서', '리건물이름', '동호', '전체주소'])

In [ ]:
# ============================================================
# 3. 데이터 구조 확인
# ============================================================

# 데이터 행과 열의 개수를 확인합니다.
print("데이터 크기:", df.shape)

# 각 컬럼의 자료형과 결측치 여부를 확인합니다.
print("\n데이터 정보:")
print(df.info())

# 정답 라벨인 도이름 컬럼의 종류와 개수를 확인합니다.
print("\n문자 라벨 개수:")
print(df["도이름"].value_counts().sort_index())

In [ ]:
X_front = df[["우편번호앞"]].values.astype(np.float32)
X_back = df[["우편번호뒤"]].values.astype(np.float32)

# 최종 입력은 [앞3자리, 뒤3자리]
X = np.concatenate([X_front, X_back], axis=1).astype(np.float32)

print("입력 예시:")
print(X[:5])

In [ ]:
# ============================================================
# 4. 라벨 인코딩
# ============================================================
sido_encoder = LabelEncoder()
sigungu_encoder = LabelEncoder()
eupmyeondong_encoder = LabelEncoder()

y_sido = sido_encoder.fit_transform(df["도이름"]).astype(np.int64)
y_sigungu = sigungu_encoder.fit_transform(df["시군구이름"]).astype(np.int64)
y_eupmyeondong = eupmyeondong_encoder.fit_transform(df["읍면동이름"]).astype(np.int64)

num_sido_classes = len(sido_encoder.classes_)
num_sigungu_classes = len(sigungu_encoder.classes_)
num_eupmyeondong_classes = len(eupmyeondong_encoder.classes_)

print("도 클래스 수:", num_sido_classes)
print("시군구 클래스 수:", num_sigungu_classes)
print("읍면동 클래스 수:", num_eupmyeondong_classes)

In [ ]:
# ============================================================
# 5. 학습/테스트 분리
# ============================================================
indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

X_train = X[train_idx]
X_test = X[test_idx]

y_sido_train = y_sido[train_idx]
y_sido_test = y_sido[test_idx]

y_sigungu_train = y_sigungu[train_idx]
y_sigungu_test = y_sigungu[test_idx]

y_eupmyeondong_train = y_eupmyeondong[train_idx]
y_eupmyeondong_test = y_eupmyeondong[test_idx]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

In [ ]:
# ============================================================
# 6. 스케일링
# ============================================================
# 앞3자리와 뒤3자리 둘 다 숫자 feature이므로 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_sido_train_tensor = torch.tensor(y_sido_train, dtype=torch.long)
y_sido_test_tensor = torch.tensor(y_sido_test, dtype=torch.long)

y_sigungu_train_tensor = torch.tensor(y_sigungu_train, dtype=torch.long)
y_sigungu_test_tensor = torch.tensor(y_sigungu_test, dtype=torch.long)

y_eupmyeondong_train_tensor = torch.tensor(y_eupmyeondong_train, dtype=torch.long)
y_eupmyeondong_test_tensor = torch.tensor(y_eupmyeondong_test, dtype=torch.long)

In [ ]:
# ============================================================
# 7. DataLoader
# ============================================================
train_dataset = TensorDataset(
    X_train_tensor,
    y_sido_train_tensor,
    y_sigungu_train_tensor,
    y_eupmyeondong_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

print("배치 수:", len(train_loader))

In [ ]:
# ============================================================
# 6. 멀티클래스 hinge loss
#    원본 노트북의 MultiClassHingeLoss 구조 유지
# ============================================================
class MultiClassHingeLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, scores, targets):
        batch_size = scores.size(0)

        correct_scores = scores[torch.arange(batch_size), targets].view(-1, 1)
        margins = torch.clamp(scores - correct_scores + self.margin, min=0.0)
        margins[torch.arange(batch_size), targets] = 0.0

        loss = margins.sum(dim=1).mean()
        return loss

In [ ]:
# ============================================================
# 7. 선형 SVM 기반 멀티아웃풋 분류기
#    - 앞 3자리 -> 도, 시군구
#    - 뒤 3자리 -> 읍면동
# ============================================================
class MultiOutputLinearSVMClassifier(nn.Module):
    def __init__(self, num_sido_classes, num_sigungu_classes, num_eupmyeondong_classes):
        super().__init__()

        # 앞 3자리 전용 선형 SVM score layer
        self.sido_head = nn.Linear(1, num_sido_classes)
        self.sigungu_head = nn.Linear(1, num_sigungu_classes)

        # 뒤 3자리 전용 선형 SVM score layer
        self.eupmyeondong_head = nn.Linear(1, num_eupmyeondong_classes)

    def forward(self, x):
        x_front = x[:, 0:1]
        x_back = x[:, 1:2]

        sido_scores = self.sido_head(x_front)
        sigungu_scores = self.sigungu_head(x_front)
        eupmyeondong_scores = self.eupmyeondong_head(x_back)

        return sido_scores, sigungu_scores, eupmyeondong_scores

In [ ]:
# ============================================================
# 8. 학습 함수
#    원본 노트북 trainmodel 스타일 유지
# ============================================================
def trainmodel(model, trainloader, criterion, optimizer, epochs=30, l2lambda=0.0):
    loss_history = []

    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batchX, batch_sido, batch_sigungu, batch_eupmyeondong in trainloader:
            batchX = batchX.to(device)
            batch_sido = batch_sido.to(device)
            batch_sigungu = batch_sigungu.to(device)
            batch_eupmyeondong = batch_eupmyeondong.to(device)

            optimizer.zero_grad()

            sido_scores, sigungu_scores, eupmyeondong_scores = model(batchX)

            loss_sido = criterion(sido_scores, batch_sido)
            loss_sigungu = criterion(sigungu_scores, batch_sigungu)
            loss_eupmyeondong = criterion(eupmyeondong_scores, batch_eupmyeondong)

            loss = loss_sido + loss_sigungu + loss_eupmyeondong

            if l2lambda > 0:
                l2norm = sum(torch.sum(param ** 2) for param in model.parameters())
                loss = loss + l2lambda * l2norm

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(trainloader)
        loss_history.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return loss_history

In [ ]:
# # ============================================================
# # 10. 모델 학습 함수 정의
# # ============================================================

# def train_model(model, train_loader, criterion, optimizer, epochs=30, l2_lambda=0.0):
#     # model: 학습할 PyTorch 모델입니다.
#     # train_loader: 미니배치 단위로 데이터를 공급하는 DataLoader입니다.
#     # criterion: 손실 함수입니다.
#     # optimizer: 모델 파라미터를 업데이트하는 최적화 알고리즘입니다.
#     # epochs: 전체 데이터를 몇 번 반복 학습할지 정합니다.
#     # l2_lambda: SVM의 C와 반대 성격에 가까운 L2 규제 강도입니다.

#     # epoch별 평균 손실을 저장할 리스트입니다.
#     loss_history = []

#     # 모델을 학습 모드로 전환합니다.
#     model.train()

#     # 지정된 epoch 수만큼 반복합니다.
#     for epoch in range(epochs):
#         # 한 epoch 동안의 손실 합계를 저장합니다.
#         total_loss = 0.0

#         # DataLoader에서 미니배치 데이터를 하나씩 꺼냅니다.
#         for batch_X, batch_y in train_loader:
#             # 입력 데이터를 GPU 또는 CPU 장치로 이동합니다.
#             batch_X = batch_X.to(device)

#             # 정답 데이터를 GPU 또는 CPU 장치로 이동합니다.
#             batch_y = batch_y.to(device)

#             # 이전 배치에서 계산된 기울기를 초기화합니다.
#             optimizer.zero_grad()

#             # 모델에 입력 데이터를 넣어 클래스별 점수를 계산합니다.
#             scores = model(batch_X)

#             # SVM hinge loss를 계산합니다.
#             loss = criterion(scores, batch_y)

#             # L2 규제 항을 추가합니다.
#             # SVM은 마진 최대화를 위해 가중치 크기를 제어하는 규제를 사용합니다.
#             if l2_lambda > 0:
#                 l2_norm = sum(torch.sum(param ** 2) for param in model.parameters())
#                 loss = loss + l2_lambda * l2_norm

#             # 손실에 대한 기울기를 자동 계산합니다.
#             loss.backward()

#             # 계산된 기울기를 이용해 모델 파라미터를 업데이트합니다.
#             optimizer.step()

#             # 현재 배치 손실을 누적합니다.
#             total_loss += loss.item()

#         # 한 epoch의 평균 손실을 계산합니다.
#         avg_loss = total_loss / len(train_loader)

#         # 평균 손실을 리스트에 저장합니다.
#         loss_history.append(avg_loss)

#         # 5 epoch마다 학습 상황을 출력합니다.
#         if (epoch + 1) % 5 == 0:
#             print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

#     # 학습 손실 기록을 반환합니다.
#     return loss_history

In [ ]:
# ============================================================
# 9. 평가 함수
#    - 도 정확도
#    - 시군구 정확도
#    - 읍면동 정확도
#    - 세 개 모두 맞춘 exact match accuracy
# ============================================================
def evaluatemodel(model, Xtensor, y_sido_tensor, y_sigungu_tensor, y_eupmyeondong_tensor):
    model.eval()

    with torch.no_grad():
        Xtensor = Xtensor.to(device)

        sido_scores, sigungu_scores, eupmyeondong_scores = model(Xtensor)

        pred_sido = torch.argmax(sido_scores, dim=1).cpu().numpy()
        pred_sigungu = torch.argmax(sigungu_scores, dim=1).cpu().numpy()
        pred_eupmyeondong = torch.argmax(eupmyeondong_scores, dim=1).cpu().numpy()

    true_sido = y_sido_tensor.cpu().numpy()
    true_sigungu = y_sigungu_tensor.cpu().numpy()
    true_eupmyeondong = y_eupmyeondong_tensor.cpu().numpy()

    sido_acc = accuracy_score(true_sido, pred_sido)
    sigungu_acc = accuracy_score(true_sigungu, pred_sigungu)
    eupmyeondong_acc = accuracy_score(true_eupmyeondong, pred_eupmyeondong)

    exact_match_acc = np.mean(
        (pred_sido == true_sido) &
        (pred_sigungu == true_sigungu) &
        (pred_eupmyeondong == true_eupmyeondong)
    )

    return {
        "sido_acc": sido_acc,
        "sigungu_acc": sigungu_acc,
        "eupmyeondong_acc": eupmyeondong_acc,
        "exact_match_acc": exact_match_acc,
        "pred_sido": pred_sido,
        "pred_sigungu": pred_sigungu,
        "pred_eupmyeondong": pred_eupmyeondong
    }

In [ ]:
# # ============================================================
# # 11. 평가 함수 정의
# # ============================================================

# def evaluate_model(model, X_tensor, y_tensor):
#     # model: 평가할 PyTorch 모델입니다.
#     # X_tensor: 평가 입력 데이터입니다.
#     # y_tensor: 실제 정답 데이터입니다.

#     # 모델을 평가 모드로 전환합니다.
#     model.eval()

#     # 평가에서는 기울기를 계산할 필요가 없으므로 no_grad를 사용합니다.
#     with torch.no_grad():
#         # 입력 데이터를 장치로 이동합니다.
#         X_tensor = X_tensor.to(device)

#         # 모델이 클래스별 점수를 출력합니다.
#         scores = model(X_tensor)

#         # 가장 높은 점수를 가진 클래스 번호를 예측값으로 선택합니다.
#         preds = torch.argmax(scores, dim=1)

#         # 예측값을 CPU numpy 배열로 변환합니다.
#         preds_np = preds.cpu().numpy()

#         # 실제 정답도 numpy 배열로 변환합니다.
#         y_np = y_tensor.cpu().numpy()

#         # 정확도를 계산합니다.
#         acc = accuracy_score(y_np, preds_np)

#     # 정확도와 예측 결과를 반환합니다.
#     return acc, preds_np

In [ ]:
# ============================================================
# 10. 모델 학습
# ============================================================
multisvm = MultiOutputLinearSVMClassifier(
    num_sido_classes=num_sido_classes,
    num_sigungu_classes=num_sigungu_classes,
    num_eupmyeondong_classes=num_eupmyeondong_classes
).to(device)

criterion = MultiClassHingeLoss(margin=1.0)

optimizer = torch.optim.Adam(
    multisvm.parameters(),
    lr=0.01
)

loss_history = trainmodel(
    model=multisvm,
    trainloader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=50,
    l2lambda=1e-4
)

In [ ]:
# ============================================================
# 11. 평가
# ============================================================
result = evaluatemodel(
    multisvm,
    X_test_tensor,
    y_sido_test_tensor,
    y_sigungu_test_tensor,
    y_eupmyeondong_test_tensor
)

print("도 정확도         :", round(result["sido_acc"], 4))
print("시군구 정확도     :", round(result["sigungu_acc"], 4))
print("읍면동 정확도     :", round(result["eupmyeondong_acc"], 4))
print("전체 3개 동시정답 :", round(result["exact_match_acc"], 4))

In [ ]:
# ============================================================
# 14. 선형 SVM 혼동 행렬 및 분류 리포트
# ============================================================

# 혼동 행렬을 계산합니다.
cm_linear = confusion_matrix(y_test, linear_preds)

# 혼동 행렬을 DataFrame으로 변환하여 보기 쉽게 만듭니다.
cm_linear_df = pd.DataFrame(
    cm_linear,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

# 혼동 행렬을 출력합니다.
cm_linear_df

In [ ]:
print(classification_report(
    y_test,
    linear_preds,
    target_names=[str(cls) for cls in label_encoder.classes_]
))

In [ ]:
# ============================================================
# 16. RBF 특징 근사 + SVM 모델 클래스 정의
# ============================================================

class RBFFeatureSVMClassifier(nn.Module):
    # Random Fourier Features로 RBF 커널 효과를 근사하는 SVM 모델입니다.
    def __init__(self, input_dim, num_classes, rbf_dim=512, gamma=0.1):
        # 부모 클래스 초기화입니다.
        super().__init__()

        # rbf_dim은 RBF 근사 특징의 차원 수입니다.
        self.rbf_dim = rbf_dim

        # gamma는 RBF 커널의 영향 범위를 조절합니다.
        # gamma가 크면 가까운 점에만 강하게 반응하고, 작으면 넓게 반응합니다.
        self.gamma = gamma

        # Random Fourier Features에서 사용할 무작위 가중치 W입니다.
        # RBF 커널 근사를 위해 정규분포에서 샘플링합니다.
        W = torch.randn(input_dim, rbf_dim) * np.sqrt(2 * gamma)

        # 학습되지 않는 버퍼로 등록합니다.
        # 모델과 함께 device 이동은 되지만 optimizer가 업데이트하지 않습니다.
        self.register_buffer("W", W)

        # 무작위 위상 b입니다.
        b = 2 * np.pi * torch.rand(rbf_dim)

        # b도 학습되지 않는 버퍼로 등록합니다.
        self.register_buffer("b", b)

        # 변환된 RBF 근사 특징을 클래스 점수로 바꾸는 선형 계층입니다.
        self.linear = nn.Linear(rbf_dim, num_classes)

    def rbf_features(self, x):
        # 입력 x를 Random Fourier Features로 변환합니다.

        # x @ W + b를 계산합니다.
        projection = x @ self.W + self.b

        # cos 변환을 적용하여 비선형 특징을 만듭니다.
        z = torch.cos(projection)

        # 특징 크기를 안정적으로 맞추기 위해 sqrt(2 / rbf_dim)를 곱합니다.
        z = z * np.sqrt(2.0 / self.rbf_dim)

        # 변환된 특징을 반환합니다.
        return z

    def forward(self, x):
        # 입력 x를 RBF 근사 특징으로 변환합니다.
        z = self.rbf_features(x)

        # 변환된 특징으로 클래스별 점수를 계산합니다.
        scores = self.linear(z)

        # 클래스별 점수를 반환합니다.
        return scores

In [ ]:
# ============================================================
# 17. RBF 특징 SVM 모델 생성 및 학습
# ============================================================

# RBF 특징 SVM 모델 객체를 생성합니다.
rbf_svm = RBFFeatureSVMClassifier(
    input_dim=num_features,     # 원본 입력 특징 개수입니다.
    num_classes=num_classes,    # 클래스 개수입니다.
    rbf_dim=1024,               # RBF 근사 특징 차원입니다.
    gamma=0.05                  # RBF 영향 범위 조절값입니다.
).to(device)

# RBF 모델용 optimizer를 생성합니다.
rbf_optimizer = torch.optim.Adam(
    rbf_svm.parameters(),       # 학습할 파라미터입니다.
    lr=0.01                     # 학습률입니다.
)

# RBF 특징 SVM 모델을 학습합니다.
rbf_loss_history = train_model(
    model=rbf_svm,              # 학습할 모델입니다.
    train_loader=train_loader,  # 훈련 데이터 로더입니다.
    criterion=criterion,        # SVM hinge loss입니다.
    optimizer=rbf_optimizer,    # 최적화 알고리즘입니다.
    epochs=50,                  # 전체 데이터를 50번 반복 학습합니다.
    l2_lambda=1e-4              # L2 규제 강도입니다.
)

In [ ]:
# ============================================================
# 18. RBF 특징 SVM 평가
# ============================================================

# 테스트 데이터로 RBF 특징 SVM 모델 성능을 평가합니다.
rbf_acc, rbf_preds = evaluate_model(rbf_svm, X_test_tensor, y_test_tensor)

# 정확도를 출력합니다.
print(f"RBF 특징 SVM 테스트 정확도: {rbf_acc:.4f}")

# 숫자 예측값을 문자 라벨로 다시 변환합니다.
rbf_pred_letters = label_encoder.inverse_transform(rbf_preds)

# 예측 결과 일부를 출력합니다.
print("예측 문자 일부:", rbf_pred_letters[:20])

In [ ]:
# ============================================================
# 19. RBF 특징 SVM 혼동 행렬
# ============================================================

# 혼동 행렬을 계산합니다.
cm_rbf = confusion_matrix(y_test, rbf_preds)

# 혼동 행렬을 DataFrame으로 변환합니다.
cm_rbf_df = pd.DataFrame(
    cm_rbf,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

# 혼동 행렬을 출력합니다.
cm_rbf_df

In [ ]:
# ============================================================
# 20. RBF 특징 SVM 분류 리포트 출력
# ============================================================

# precision, recall, f1-score를 포함한 분류 성능 리포트를 출력합니다.
print(classification_report(
    y_test,
    rbf_preds,
    target_names=[str(cls) for cls in label_encoder.classes_]
))

In [ ]:
# ============================================================
# 21. 학습 손실 그래프
# ============================================================

# 그래프 크기를 설정합니다.
plt.figure(figsize=(10, 5))

# 선형 SVM 손실 곡선을 그립니다.
plt.plot(linear_loss_history, label="Linear SVM Loss")

# RBF 특징 SVM 손실 곡선을 그립니다.
plt.plot(rbf_loss_history, label="RBF Feature SVM Loss")

# 그래프 제목을 설정합니다.
plt.title("SVM Training Loss")

# x축 이름을 설정합니다.
plt.xlabel("Epoch")

# y축 이름을 설정합니다.
plt.ylabel("Hinge Loss")

# 범례를 표시합니다.
plt.legend()

# 격자선을 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()

In [ ]:
# ============================================================
# 22. 선형 SVM과 RBF 특징 SVM 성능 비교
# ============================================================

# 결과를 DataFrame으로 정리합니다.
result_df = pd.DataFrame({
    "Model": ["Linear SVM", "RBF Feature SVM"],
    "Test Accuracy": [linear_acc, rbf_acc]
})

# 결과를 출력합니다.
result_df

In [ ]:
# ============================================================
# 23. 예측 결과 샘플 비교
# ============================================================

# 실제 문자 라벨을 복원합니다.
true_letters = label_encoder.inverse_transform(y_test)

# 비교할 샘플 개수를 지정합니다.
sample_count = 30

# 실제값, 선형 SVM 예측값, RBF SVM 예측값을 표로 정리합니다.
compare_df = pd.DataFrame({
    "Actual": true_letters[:sample_count],
    "Linear_SVM_Pred": linear_pred_letters[:sample_count],
    "RBF_Feature_SVM_Pred": rbf_pred_letters[:sample_count]
})

# 비교 결과를 출력합니다.
compare_df

In [ ]:
import numpy as np
import pandas as pd

def sorted_confusions_from_cm(cm, class_names, exclude_diagonal=True):
    """
    cm: confusion matrix (numpy array or pandas DataFrame)
    class_names: list/array of class labels in order
    exclude_diagonal: True면 정답 맞춘 값(cm[i,i]) 제외하고 혼동만 계산
    """
    if isinstance(cm, pd.DataFrame):
        cm = cm.values

    records = []
    n = len(class_names)

    for i in range(n):
        row = cm[i].copy()

        if exclude_diagonal:
            row[i] = 0

        total_misclassified = row.sum()

        if total_misclassified == 0:
            continue

        # 해당 실제 문자에서 가장 많이 잘못 예측된 문자들 추출
        for j in np.argsort(row)[::-1]:
            if row[j] == 0:
                break
            records.append({
                "actual": class_names[i],
                "predicted": class_names[j],
                "count": int(row[j]),
                "actual_total_misclassified": int(total_misclassified),
                "misclass_rate_in_row": row[j] / total_misclassified
            })

    result = pd.DataFrame(records)
    result = result.sort_values(
        by=["count", "misclass_rate_in_row"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return result

# 예시: Linear SVM confusion matrix
cm_sorted_linear = sorted_confusions_from_cm(cm_linear, label_encoder.classes_)
display(cm_sorted_linear.head(20))

# 예시: RBF SVM confusion matrix
cm_sorted_rbf = sorted_confusions_from_cm(cm_rbf, label_encoder.classes_)
display(cm_sorted_rbf.head(20))

In [ ]:
# cm_sorted_linear 또는 cm_sorted_rbf 같은 결과 DataFrame 기준
# columns: actual, predicted, count, actual_total_misclassified, misclass_rate_in_row

merged_actual_linear = (
    cm_sorted_linear
    .groupby('actual', as_index=False)
    .agg({
        'count': 'sum',
        'actual_total_misclassified': 'first'
    })
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

display(merged_actual)

In [ ]:
# cm_sorted_linear 또는 cm_sorted_rbf 같은 결과 DataFrame 기준
# columns: actual, predicted, count, actual_total_misclassified, misclass_rate_in_row

merged_actual_rbf = (
    cm_sorted_rbf
    .groupby('actual', as_index=False)
    .agg({
        'count': 'sum',
        'actual_total_misclassified': 'first'
    })
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

display(merged_actual)